# Explainability and Error Analysis (FashionStyle14)

## Objective
This notebook runs two analysis tracks on trained fashion style classifiers:

- **Part A - Explainability (cross-attention model only):** attention-weight visualization, per-class attention patterns, and modality contribution (text vs image ablation).
- **Part B - Error analysis:** confusion matrices for all baseline models (excluding the proposed cross-attention model), class-pair confusion for cross-attention and image-only models, and Grad-CAM on CLIP vision encoders.

## Scope
| Track | Models |
|-------|--------|
| Part A | `cross_attention` (proposed multimodal fusion) |
| Part B.1 - all baselines | image-only, multimodal, text-only (**excludes proposed `cross_attention`**) |
| Part B.2 - class-pair confusion | `cross_attention` + image-only (`clip`, `convnext`, `swin`, `vit`) |
| Part B.3 - Grad-CAM | `clip` (image-only) + `cross_attention` (CLIP ViT-B/32 vision encoder) |

## Inputs (explicit paths)
| Resource | Path |
|----------|------|
| Image list | `FashionStyle14_v1/complete_dataset.csv` |
| Captions | `FashionStyle14_v1/caption/fashion_captions_llava_success.csv` |
| Split seeds | `FashionStyle14_v1/seeds_list.txt` (first 10 seeds) |
| Images | `FashionStyle14_v1/dataset/` |
| Cross-attention checkpoints | `results/proposed/cross_attention/seed_<seed>/best_model.pt` |
| Image-only checkpoints | `results/image_only/<model>/seed_<seed>/best_model.pt` |
| Multimodal checkpoints | `results/multimodal/<model>/seed_<seed>/best_model.pt` |
| Text-only checkpoints | `results/text_only/fashionbert/seed_<seed>/best_model.pt` |

Checkpoint files are referenced directly. If a file is missing at runtime, loading is skipped with a warning and weights remain randomly initialized (for pipeline smoke tests).

## Outputs
All artifacts are written under **`results/explainability/`** with one subfolder per technique:

| Subfolder | Contents |
|-----------|----------|
| `attention_weights/` | Sample-level image/text token attention bar plots |
| `per_class_attention/` | Mean cross-attention maps per style class |
| `modality_contribution/` | Confidence drop when removing image or text |
| `confusion_matrix/` | Per-model heatmaps, top confused pairs, global summary |
| `class_pair_confusion/` | Symmetric confusion and top pairs (cross-attention + image-only) |
| `gradcam/` | Success/failure Grad-CAM overlays on fashion images |

## Notes
- Explainability (Part A) and Grad-CAM (Part B.3) run on **all 10 robustness seeds**; confusion analysis (Part B.1/B.2) also aggregates all seeds.


## 1. Configuration, imports, and paths


In [ ]:
from __future__ import annotations

import json
import os
import random
import re
import warnings
from pathlib import Path
from typing import Any, Callable, Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from transformers import (
    AutoModel,
    AutoTokenizer,
    BlipModel,
    BlipProcessor,
    CLIPModel,
    CLIPProcessor,
    ViltModel,
    ViltProcessor,
)

warnings.filterwarnings("ignore", category=UserWarning)
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

# Hyperparameters aligned with training notebooks
BATCH_SIZE = 8
MAX_SEQ_LENGTH = 128
MAX_SEQ_LENGTH_VILT = 40
DROPOUT = 0.5
MODEL_INIT_SEED = 42
TRAIN_RATIO, VAL_RATIO, TEST_RATIO = 0.7, 0.15, 0.15
NUM_SEEDS_TO_USE = 10
NUM_VIZ_SAMPLES_PER_SEED = 8  # sample-level attention plots per seed (set to None for entire test set)
TOP_K_CONFUSED_PAIRS = 15
GRADCAM_NUM_CASES = 6

CLIP_MODEL_ID = "openai/clip-vit-base-patch32"
BERT_MODEL_ID = "bert-base-uncased"
BLIP_MODEL_ID = "Salesforce/blip-image-captioning-base"
VILT_MODEL_ID = "dandelin/vilt-b32-mlm"
VIT_MODEL_NAME = "vit_base_patch16_224"
CONVNEXT_MODEL_NAME = "convnext_base"
SWIN_MODEL_NAME = "swin_base_patch4_window7_224"

CROSS_ATTN_DIM = 512
CROSS_ATTN_HEADS = 8
CROSS_ATTN_DROPOUT = 0.1

IMAGE_ONLY_MODELS = ["clip", "convnext", "swin", "vit"]
MULTIMODAL_MODELS = ["concat_mlp", "gated_fusion", "blip", "vilbert"]
TEXT_ONLY_MODELS = ["fashionbert"]
PROPOSED_MODEL_KEY = "cross_attention"  # excluded from Part B.1 "all models"

ALL_BASELINE_MODELS: Dict[str, Tuple[str, str]] = {
    **{m: ("image_only", m) for m in IMAGE_ONLY_MODELS},
    **{m: ("multimodal", m) for m in MULTIMODAL_MODELS},
    **{m: ("text_only", m) for m in TEXT_ONLY_MODELS},
}
CLASS_PAIR_MODELS = [PROPOSED_MODEL_KEY] + IMAGE_ONLY_MODELS
# Grad-CAM uses the CLIP ViT vision encoder (clip image-only + cross-attention)
GRADCAM_MODELS = ["clip", PROPOSED_MODEL_KEY]

OUTPUT_ROOT = Path("results/explainability")
DIR_ATTENTION = OUTPUT_ROOT / "attention_weights"
DIR_PER_CLASS_ATTN = OUTPUT_ROOT / "per_class_attention"
DIR_MODALITY = OUTPUT_ROOT / "modality_contribution"
DIR_CONFUSION = OUTPUT_ROOT / "confusion_matrix"
DIR_CLASS_PAIR = OUTPUT_ROOT / "class_pair_confusion"
DIR_GRADCAM = OUTPUT_ROOT / "gradcam"
for d in [DIR_ATTENTION, DIR_PER_CLASS_ATTN, DIR_MODALITY, DIR_CONFUSION, DIR_CLASS_PAIR, DIR_GRADCAM]:
    d.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("Output root:", OUTPUT_ROOT.resolve())
print("Baseline models (Part B.1):", list(ALL_BASELINE_MODELS.keys()))
print("Excluded proposed model:", PROPOSED_MODEL_KEY)

## 2. Dataset loading and split helpers


In [ ]:
def resolve_paths() -> Tuple[Path, Path, Path]:
    cwd = Path.cwd().resolve()
    for root in [cwd, cwd / "FusionStyle", cwd.parent / "FusionStyle"]:
        data_dir = root / "FashionStyle14_v1"
        if (data_dir / "complete_dataset.csv").is_file():
            return root, data_dir, data_dir / "caption" / "fashion_captions_llava_success.csv"
    raise FileNotFoundError("FashionStyle14_v1 not found; run from FusionStyle/.")


def load_seeds(seeds_file: Path) -> List[int]:
    content = seeds_file.read_text(encoding="utf-8")
    matches = re.findall(r"Seed\s+(\d+)", content, flags=re.IGNORECASE)
    return sorted({int(s) for s in matches})[:NUM_SEEDS_TO_USE]


def normalize_rel_path(path_str: str) -> str:
    return str(path_str).strip().replace("\\", "/")


def canonical_merge_key(raw: str, image_root: Path) -> str:
    s = normalize_rel_path(raw).lstrip("./")
    low = s.lower()
    if low.startswith("fashionstyle14_v1/"):
        s = s[len("fashionstyle14_v1/") :].lstrip("/")
        low = s.lower()
    marker = "dataset/"
    ix = low.find(marker)
    if ix >= 0:
        return normalize_rel_path(s[ix:])
    p = Path(s)
    if p.is_absolute():
        try:
            rel = Path(p.resolve()).relative_to(image_root.resolve())
            return normalize_rel_path(str(rel).replace(os.sep, "/"))
        except ValueError:
            pass
    return s


def load_image_only_frame(csv_path: Path, image_root: Path) -> pd.DataFrame:
    lines = csv_path.read_text(encoding="utf-8").splitlines()
    rel = [ln.strip() for ln in lines if ln.strip()]
    df = pd.DataFrame({"rel_path": rel})
    df["rel_path"] = df["rel_path"].map(normalize_rel_path)
    df["merge_key"] = df["rel_path"].map(lambda r: canonical_merge_key(r, image_root))
    df["style"] = df["merge_key"].str.split("/").str[1]
    df["abs_path"] = df["rel_path"].apply(lambda r: str((image_root / r.replace("/", os.sep)).resolve()))
    return df[df["abs_path"].map(os.path.isfile)].reset_index(drop=True)


def load_captions(path: Path, image_root: Path) -> pd.DataFrame:
    df = pd.read_csv(path, encoding="utf-8")
    if "status" in df.columns:
        df = df[df["status"].astype(str).str.lower() == "success"]
    path_col = next(c for c in df.columns if c.lower().strip() in {"image_path", "rel_path", "path", "filename"})
    cap_col = next(c for c in df.columns if "caption" in c.lower() or c.lower() in {"text", "description"})
    out = df[[path_col, cap_col]].rename(columns={path_col: "raw_image_path", cap_col: "caption"})
    out["merge_key"] = out["raw_image_path"].map(lambda r: canonical_merge_key(r, image_root))
    out["caption"] = out["caption"].fillna("").astype(str).str.strip()
    return out[out["caption"] != ""].drop_duplicates("merge_key", keep="last")


PROJECT_ROOT, DATA_DIR, CAPTION_CSV = resolve_paths()
IMAGE_ROOT = DATA_DIR
COMPLETE_CSV = DATA_DIR / "complete_dataset.csv"
SEEDS = load_seeds(DATA_DIR / "seeds_list.txt")

df_images = load_image_only_frame(COMPLETE_CSV, IMAGE_ROOT)
cap_df = load_captions(CAPTION_CSV, IMAGE_ROOT)
df_mm = df_images.merge(cap_df[["merge_key", "caption"]], on="merge_key", how="inner").reset_index(drop=True)

classes = sorted(df_images["style"].unique().tolist())
assert len(classes) == 14
style_to_idx = {s: i for i, s in enumerate(classes)}
idx_to_style = {i: s for s, i in style_to_idx.items()}
num_classes = len(classes)


def split_by_seed(df: pd.DataFrame, seed_value: int) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    train_df, temp_df = train_test_split(df, test_size=VAL_RATIO + TEST_RATIO, stratify=df["style"], random_state=seed_value)
    val_df, test_df = train_test_split(temp_df, test_size=TEST_RATIO / (VAL_RATIO + TEST_RATIO), stratify=temp_df["style"], random_state=seed_value)
    return train_df.reset_index(drop=True), val_df.reset_index(drop=True), test_df.reset_index(drop=True)


print("PROJECT_ROOT:", PROJECT_ROOT)
print("Samples (image-only):", len(df_images))
print("Samples (with captions):", len(df_mm))
print("Classes:", classes)
print("Seeds:", SEEDS)

## 3. Shared utilities (checkpoints, confusion, plotting)


In [ ]:
def checkpoint_path(category: str, model_key: str, seed: int) -> Path:
    if category == "proposed":
        return PROJECT_ROOT / "results" / "proposed" / model_key / f"seed_{seed}" / "best_model.pt"
    return PROJECT_ROOT / "results" / category / model_key / f"seed_{seed}" / "best_model.pt"


def load_weights(model: nn.Module, ckpt_path: Path) -> None:
    print(f"Loading checkpoint: {ckpt_path}")
    if not ckpt_path.is_file():
        print(f"  WARNING: checkpoint not found; using current weights (pretend path for pipeline).")
        return
    state = torch.load(ckpt_path, map_location=device, weights_only=True)
    model.load_state_dict(state, strict=False)
    print("  Loaded successfully.")


def top_confused_pairs(cm: np.ndarray, k: int = TOP_K_CONFUSED_PAIRS) -> pd.DataFrame:
    n = cm.shape[0]
    rows = []
    for i in range(n):
        for j in range(n):
            if i != j and cm[i, j] > 0:
                rows.append({
                    "true_class": idx_to_style[i],
                    "pred_class": idx_to_style[j],
                    "count": int(cm[i, j]),
                    "rate_given_true": float(cm[i, j] / max(cm[i].sum(), 1)),
                })
    df = pd.DataFrame(rows).sort_values("count", ascending=False).head(k).reset_index(drop=True)
    return df


def symmetric_confusion_rates(cm: np.ndarray) -> pd.DataFrame:
    pairs = []
    n = cm.shape[0]
    for i in range(n):
        for j in range(i + 1, n):
            a, b = cm[i, j], cm[j, i]
            if a + b == 0:
                continue
            pairs.append({
                "class_a": idx_to_style[i],
                "class_b": idx_to_style[j],
                "a_to_b": int(a),
                "b_to_a": int(b),
                "symmetric_total": int(a + b),
                "symmetric_rate": float((a + b) / max(cm.sum(), 1)),
            })
    return pd.DataFrame(pairs).sort_values("symmetric_total", ascending=False).reset_index(drop=True)


def plot_confusion_heatmap(cm: np.ndarray, title: str, out_path: Path) -> None:
    fig, ax = plt.subplots(figsize=(12, 10))
    sns.heatmap(cm, annot=False, cmap="Blues", xticklabels=classes, yticklabels=classes, ax=ax)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(title)
    plt.xticks(rotation=45, ha="right")
    plt.yticks(rotation=0)
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.show()
    plt.close(fig)
    print(f"Saved heatmap: {out_path}")



def build_classifier_head(in_dim: int, num_classes: int) -> nn.Sequential:
    """Shared MLP classifier head (512/768/etc -> 256 -> 128 -> num_classes)."""
    return nn.Sequential(
        nn.Linear(in_dim, 256),
        nn.ReLU(inplace=True),
        nn.Dropout(DROPOUT),
        nn.Linear(256, 128),
        nn.ReLU(inplace=True),
        nn.Dropout(DROPOUT),
        nn.Linear(128, num_classes),
    )


_BACKBONE_CACHE: Dict[str, Any] = {}


def get_clip_bert():
    if "clip" not in _BACKBONE_CACHE:
        _BACKBONE_CACHE["clip"] = CLIPModel.from_pretrained(CLIP_MODEL_ID)
        _BACKBONE_CACHE["clip_p"] = CLIPProcessor.from_pretrained(CLIP_MODEL_ID)
        _BACKBONE_CACHE["bert"] = AutoModel.from_pretrained(BERT_MODEL_ID)
        _BACKBONE_CACHE["bert_t"] = AutoTokenizer.from_pretrained(BERT_MODEL_ID)
    return (
        _BACKBONE_CACHE["clip"],
        _BACKBONE_CACHE["clip_p"],
        _BACKBONE_CACHE["bert"],
        _BACKBONE_CACHE["bert_t"],
    )


def _batch_labels(batch: Dict[str, Any]) -> torch.Tensor:
    """Accept batches from image-only or multimodal loaders."""
    if "labels" in batch:
        lab = batch["labels"]
    elif "label" in batch:
        lab = batch["label"]
    else:
        raise KeyError("batch must contain 'labels' or 'label'")
    if not isinstance(lab, torch.Tensor):
        lab = torch.as_tensor(lab, dtype=torch.long)
    if lab.dim() == 0:
        lab = lab.unsqueeze(0)
    return lab


def _model_batch(batch: Dict[str, Any]) -> Dict[str, Any]:
    """Forward-pass tensors only (skip metadata keys)."""
    skip = {"label", "labels", "abs_path", "abs_paths"}
    out: Dict[str, Any] = {}
    for k, v in batch.items():
        if k in skip:
            continue
        out[k] = v.to(device) if torch.is_tensor(v) else v
    return out


def predict_loader(model, loader, forward_fn) -> Tuple[np.ndarray, np.ndarray]:
    model.eval()
    ys, ps = [], []
    with torch.no_grad():
        for batch in loader:
            labels = _batch_labels(batch).to(device)
            logits = forward_fn(model, _model_batch(batch))
            pred = logits.argmax(dim=1)
            ys.extend(labels.cpu().tolist())
            ps.extend(pred.cpu().tolist())
    return np.array(ys), np.array(ps)

print("Shared utilities ready.")


## Explainability (Cross-Attention)

### Cross-attention model with attention weight export
The fusion module returns averaged multi-head attention weights from image query to text keys.


In [ ]:
class FashionMultiModalFrameDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, style_to_idx: Dict[str, int]):
        self.frame = frame.reset_index(drop=True)
        self.style_to_idx = style_to_idx

    def __len__(self) -> int:
        return len(self.frame)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        row = self.frame.iloc[idx]
        return {"abs_path": row["abs_path"], "caption": str(row["caption"]), "labels": torch.tensor(self.style_to_idx[row["style"]], dtype=torch.long)}


class CrossAttentionFusion(nn.Module):
    def __init__(self, visual_dim: int, textual_dim: int, d_model: int = CROSS_ATTN_DIM, nhead: int = CROSS_ATTN_HEADS, dropout: float = CROSS_ATTN_DROPOUT):
        super().__init__()
        self.visual_proj = nn.Linear(visual_dim, d_model)
        self.text_proj = nn.Linear(textual_dim, d_model)
        self.cross_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, visual_feat, text_tokens, key_padding_mask=None, return_attn: bool = False):
        query = self.visual_proj(visual_feat).unsqueeze(1)
        key_value = self.text_proj(text_tokens)
        attn_out, attn_w = self.cross_attn(query, key_value, key_value, key_padding_mask=key_padding_mask, need_weights=True, average_attn_weights=True)
        fused = self.norm(attn_out.squeeze(1))
        if return_attn:
            return fused, attn_w.squeeze(1)
        return fused


def build_classifier_head(in_dim: int, num_classes: int) -> nn.Sequential:
    return nn.Sequential(
        nn.Linear(in_dim, 256), nn.ReLU(inplace=True), nn.Dropout(DROPOUT),
        nn.Linear(256, 128), nn.ReLU(inplace=True), nn.Dropout(DROPOUT),
        nn.Linear(128, num_classes),
    )


_CACHE: Dict[str, Any] = {}


def get_clip_bert():
    if "clip" not in _CACHE:
        _CACHE["clip"] = CLIPModel.from_pretrained(CLIP_MODEL_ID)
        _CACHE["clip_p"] = CLIPProcessor.from_pretrained(CLIP_MODEL_ID)
        _CACHE["bert"] = AutoModel.from_pretrained(BERT_MODEL_ID)
        _CACHE["bert_t"] = AutoTokenizer.from_pretrained(BERT_MODEL_ID)
    return _CACHE["clip"], _CACHE["clip_p"], _CACHE["bert"], _CACHE["bert_t"]


class ClipBertCrossAttentionClassifier(nn.Module):
    def __init__(self, clip_model, clip_processor, bert_model, bert_tokenizer, num_classes: int):
        super().__init__()
        self.clip_model = clip_model
        self.clip_processor = clip_processor
        self.bert_model = bert_model
        self.bert_tokenizer = bert_tokenizer
        for m in [self.clip_model, self.bert_model]:
            for p in m.parameters():
                p.requires_grad = False
        self.clip_model.eval(); self.bert_model.eval()
        vd = clip_model.config.projection_dim
        td = bert_model.config.hidden_size
        self.fusion = CrossAttentionFusion(vd, td)
        self.classifier = build_classifier_head(CROSS_ATTN_DIM, num_classes)

    def encode(self, pixel_values, captions, zero_image: bool = False, zero_text: bool = False):
        dev = pixel_values.device
        with torch.no_grad():
            visual = self.clip_model.get_image_features(pixel_values=pixel_values).float()
            enc = self.bert_tokenizer(captions, return_tensors="pt", padding=True, truncation=True, max_length=MAX_SEQ_LENGTH).to(dev)
            text_tokens = self.bert_model(**enc).last_hidden_state.float()
        if zero_image:
            visual = torch.zeros_like(visual)
        if zero_text:
            text_tokens = torch.zeros_like(text_tokens)
        pad_mask = enc["attention_mask"] == 0
        return visual, text_tokens, pad_mask, enc

    def forward(self, pixel_values, captions, return_attn: bool = False, zero_image: bool = False, zero_text: bool = False):
        visual, text_tokens, pad_mask, _ = self.encode(pixel_values, captions, zero_image, zero_text)
        if return_attn:
            fused, attn = self.fusion(visual, text_tokens, pad_mask, return_attn=True)
            return self.classifier(fused), attn
        fused = self.fusion(visual, text_tokens, pad_mask)
        return self.classifier(fused)


def collate_mm(batch, clip_processor):
    images = [Image.open(x["abs_path"]).convert("RGB") for x in batch]
    pixel_values = clip_processor(images=images, return_tensors="pt")["pixel_values"]
    return {"pixel_values": pixel_values, "captions": [x["caption"] for x in batch], "labels": torch.stack([x["labels"] for x in batch])}


def build_cross_attention_model() -> ClipBertCrossAttentionClassifier:
    clip_m, clip_p, bert_m, bert_t = get_clip_bert()
    return ClipBertCrossAttentionClassifier(clip_m, clip_p, bert_m, bert_t, num_classes)

print("Cross-attention model defined.")


### 1 Attention-weight visualization
Visualize **image token (query)** attention over **text tokens (keys)** for sample outfits. Runs for **every seed** in `SEEDS`; outputs go to `attention_weights/seed_<seed>/`.


In [ ]:
clip_p = get_clip_bert()[1]
all_attn_records = []

for seed in SEEDS:
    print("\n" + "=" * 60)
    print(f"A.1 Attention visualization | seed {seed}")
    seed_dir = DIR_ATTENTION / f"seed_{seed}"
    seed_dir.mkdir(parents=True, exist_ok=True)

    _, _, test_df = split_by_seed(df_mm, seed)
    test_loader = DataLoader(
        FashionMultiModalFrameDataset(test_df, style_to_idx),
        batch_size=1,
        shuffle=False,
        collate_fn=lambda b: collate_mm(b, clip_p),
    )

    ca_model = build_cross_attention_model().to(device)
    load_weights(ca_model, checkpoint_path("proposed", PROPOSED_MODEL_KEY, seed))
    ca_model.eval()

    sample_records = []
    num_viz = len(test_loader) if NUM_VIZ_SAMPLES_PER_SEED is None else min(NUM_VIZ_SAMPLES_PER_SEED, len(test_loader))

    for bi, batch in enumerate(test_loader):
        if bi >= num_viz:
            break
        pv = batch["pixel_values"].to(device)
        caps = batch["captions"]
        label = int(batch["labels"].item())
        with torch.no_grad():
            logits, attn_w = ca_model(pv, caps, return_attn=True)
        probs = F.softmax(logits, dim=1)[0]
        pred = int(probs.argmax().item())
        conf = float(probs[pred].item())

        enc = ca_model.bert_tokenizer(caps, return_tensors="pt", padding=True, truncation=True, max_length=MAX_SEQ_LENGTH)
        tokens = ca_model.bert_tokenizer.convert_ids_to_tokens(enc["input_ids"][0].tolist())
        weights = attn_w[0].cpu().numpy()
        valid_len = int((enc["attention_mask"][0] == 1).sum().item())
        tokens = tokens[:valid_len]
        weights = weights[:valid_len]

        fig, ax = plt.subplots(figsize=(10, 3))
        ax.bar(range(len(weights)), weights, color="steelblue")
        ax.set_xticks(range(len(tokens)))
        ax.set_xticklabels(tokens, rotation=60, ha="right", fontsize=8)
        ax.set_ylabel("Attention weight")
        ax.set_title(f"seed {seed} | true={idx_to_style[label]} pred={idx_to_style[pred]} conf={conf:.3f}")
        fig.tight_layout()
        out_png = seed_dir / f"sample_{bi:02d}_text_token_attention.png"
        fig.savefig(out_png, dpi=150)
        plt.show()
        plt.close(fig)

        fig2, ax2 = plt.subplots(figsize=(10, 1.5))
        ax2.imshow(weights[None, :], aspect="auto", cmap="viridis")
        ax2.set_yticks([0])
        ax2.set_yticklabels(["image query"])
        ax2.set_xticks(range(len(tokens)))
        ax2.set_xticklabels(tokens, rotation=60, ha="right", fontsize=8)
        ax2.set_title("Image-token (query) attention over text tokens")
        fig2.tight_layout()
        out_heat = seed_dir / f"sample_{bi:02d}_image_query_heatmap.png"
        fig2.savefig(out_heat, dpi=150)
        plt.show()
        plt.close(fig2)

        row = {
            "seed": seed,
            "sample_idx": bi,
            "true": idx_to_style[label],
            "pred": idx_to_style[pred],
            "confidence": conf,
            "max_attn_token": tokens[int(weights.argmax())],
        }
        sample_records.append(row)
        print(row)

    df_seed = pd.DataFrame(sample_records)
    df_seed.to_csv(seed_dir / "sample_attention_summary.csv", index=False)
    all_attn_records.extend(sample_records)

df_attn_samples = pd.DataFrame(all_attn_records)
df_attn_samples.to_csv(DIR_ATTENTION / "all_seeds_sample_attention_summary.csv", index=False)
print(df_attn_samples)
print(f"Saved all seeds summary: {DIR_ATTENTION / 'all_seeds_sample_attention_summary.csv'}")


### 2 Per-class attention patterns
Average cross-attention weights over all test samples **per true style class**, computed for **every seed**. Per-seed outputs: `per_class_attention/seed_<seed>/`; combined summary: `per_class_attention/all_seeds_summary.csv`.


In [ ]:
max_tokens = MAX_SEQ_LENGTH
clip_p = get_clip_bert()[1]
all_pc_rows = []

for seed in SEEDS:
    print("\n" + "=" * 60)
    print(f"A.2 Per-class attention | seed {seed}")
    seed_dir = DIR_PER_CLASS_ATTN / f"seed_{seed}"
    seed_dir.mkdir(parents=True, exist_ok=True)

    _, _, test_df = split_by_seed(df_mm, seed)
    ca_model = build_cross_attention_model().to(device)
    load_weights(ca_model, checkpoint_path("proposed", PROPOSED_MODEL_KEY, seed))
    ca_model.eval()

    class_attn_sum = np.zeros((num_classes, max_tokens), dtype=np.float64)
    class_counts = np.zeros(num_classes, dtype=np.int64)

    with torch.no_grad():
        for batch in DataLoader(
            FashionMultiModalFrameDataset(test_df, style_to_idx),
            batch_size=BATCH_SIZE,
            shuffle=False,
            collate_fn=lambda b: collate_mm(b, clip_p),
        ):
            labels = batch["labels"].numpy()
            pv = batch["pixel_values"].to(device)
            _, attn_w = ca_model(pv, batch["captions"], return_attn=True)
            attn_np = attn_w.cpu().numpy()
            for i, lab in enumerate(labels):
                w = attn_np[i]
                if w.shape[0] < max_tokens:
                    w = np.pad(w, (0, max_tokens - w.shape[0]))
                else:
                    w = w[:max_tokens]
                class_attn_sum[lab] += w
                class_counts[lab] += 1

    class_attn_mean = class_attn_sum / np.maximum(class_counts[:, None], 1)
    np.save(seed_dir / "per_class_mean_attention.npy", class_attn_mean)

    fig, ax = plt.subplots(figsize=(14, 8))
    sns.heatmap(class_attn_mean, xticklabels=range(max_tokens), yticklabels=classes, cmap="viridis", ax=ax)
    ax.set_xlabel("Text token position")
    ax.set_ylabel("True style class")
    ax.set_title(f"Mean cross-attention per class | seed {seed}")
    fig.tight_layout()
    fig.savefig(seed_dir / "per_class_attention_heatmap.png", dpi=150)
    plt.show()
    plt.close(fig)

    for ci, cls in enumerate(classes):
        peak_pos = int(class_attn_mean[ci].argmax())
        all_pc_rows.append({
            "seed": seed,
            "class": cls,
            "n_samples": int(class_counts[ci]),
            "peak_token_pos": peak_pos,
            "peak_weight": float(class_attn_mean[ci, peak_pos]),
        })

df_pc = pd.DataFrame(all_pc_rows)
df_pc.to_csv(DIR_PER_CLASS_ATTN / "all_seeds_summary.csv", index=False)
print(df_pc)
print(f"Saved all seeds summary: {DIR_PER_CLASS_ATTN / 'all_seeds_summary.csv'}")


### 3 Modality contribution analysis
Compare prediction confidence under **full**, **text removed**, and **image removed** for **every seed**. Per-seed CSVs: `modality_contribution/seed_<seed>/`; combined: `modality_contribution/all_seeds_summary.csv`.


In [ ]:
def eval_modality_setting(model, test_df, zero_image: bool, zero_text: bool) -> pd.DataFrame:
    rows = []
    clip_p = get_clip_bert()[1]
    with torch.no_grad():
        for batch in DataLoader(
            FashionMultiModalFrameDataset(test_df, style_to_idx),
            batch_size=BATCH_SIZE,
            shuffle=False,
            collate_fn=lambda b: collate_mm(b, clip_p),
        ):
            labels = batch["labels"].to(device)
            pv = batch["pixel_values"].to(device)
            logits = model(pv, batch["captions"], zero_image=zero_image, zero_text=zero_text)
            probs = F.softmax(logits, dim=1)
            conf = probs.max(dim=1).values
            pred = probs.argmax(dim=1)
            for i in range(labels.size(0)):
                rows.append({
                    "true": int(labels[i].item()),
                    "pred": int(pred[i].item()),
                    "confidence": float(conf[i].item()),
                    "correct": bool(pred[i].item() == labels[i].item()),
                })
    return pd.DataFrame(rows)


settings = {
    "full": (False, False),
    "remove_text": (False, True),
    "remove_image": (True, False),
}
all_summary_rows = []

for seed in SEEDS:
    print("\n" + "=" * 60)
    print(f"A.3 Modality contribution | seed {seed}")
    seed_dir = DIR_MODALITY / f"seed_{seed}"
    seed_dir.mkdir(parents=True, exist_ok=True)

    _, _, test_df = split_by_seed(df_mm, seed)
    ca_model = build_cross_attention_model().to(device)
    load_weights(ca_model, checkpoint_path("proposed", PROPOSED_MODEL_KEY, seed))
    ca_model.eval()

    summary_rows = []
    for name, (zi, zt) in settings.items():
        df_s = eval_modality_setting(ca_model, test_df, zi, zt)
        df_s.to_csv(seed_dir / f"per_sample_{name}.csv", index=False)
        summary_rows.append({
            "seed": seed,
            "setting": name,
            "mean_confidence": float(df_s["confidence"].mean()),
            "accuracy": float(df_s["correct"].mean()),
            "n_samples": len(df_s),
        })
        print(f"  [{name}] mean confidence={df_s['confidence'].mean():.4f} accuracy={df_s['correct'].mean():.4f}")

    df_seed = pd.DataFrame(summary_rows)
    full_conf = df_seed.loc[df_seed["setting"] == "full", "mean_confidence"].iloc[0]
    df_seed["confidence_drop_vs_full"] = full_conf - df_seed["mean_confidence"]
    df_seed.to_csv(seed_dir / "modality_contribution_summary.csv", index=False)
    all_summary_rows.extend(summary_rows)

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(df_seed["setting"], df_seed["mean_confidence"], color=["#4c72b0", "#dd8452", "#55a868"])
    ax.set_ylabel("Mean max-class confidence")
    ax.set_title(f"Modality contribution | seed {seed}")
    fig.tight_layout()
    fig.savefig(seed_dir / "modality_confidence_bar.png", dpi=150)
    plt.show()
    plt.close(fig)

df_mod = pd.DataFrame(all_summary_rows)
df_mod.to_csv(DIR_MODALITY / "all_seeds_summary.csv", index=False)
print(df_mod)
print(f"Saved all seeds summary: {DIR_MODALITY / 'all_seeds_summary.csv'}")
